# 02 Piston source: a baffled circular piston and a bone sphere

| | |
|---|---|
| **Purpose** | Replace the plane wave with a focused-ultrasound style transducer: a plane circular piston. Shows how a source is placed and aimed, and how OptimUS builds its field from point sources. |
| **Inputs** | 1 kHz piston, radius 1 m, centre at (-4, 0, 0), axis +x, normal velocity 1 m/s, 6 point sources per wavelength (default); bone sphere of radius 1 m at the origin, element size 0.4 m, in water. |
| **Outputs** | Incident and total pressure on the plane z = 0 (`outputs/02_piston_source_*.png`); on-axis comparison plot against the analytical formula. |
| **Checks** | On-axis incident pressure matches the analytical piston formula, and the error falls when the source is sampled more finely; GMRES converged; field finite. |
| **Runtime** | About 25 s full, 17 s CI mode (Docker image, 4 CPUs, 9 Nov 2026). CI mode uses a coarser plotting grid. |
| **Adapted from** | `notebooks/Piston source.ipynb` (upstream OptimUS). |

## Physics notes

OptimUS models the piston as a baffled radiator: the face is covered with point sources and the
field is the discrete Rayleigh integral
$p(\mathbf{x}) = \frac{i \omega \rho_0 U_0}{2 \pi} \int_S \frac{e^{i k r}}{r} \, dS$.
On the axis the integral has a closed form (Kinsler et al., *Fundamentals of Acoustics*, 4th ed., 2000, section 7.4):

$$ |p(z)| = 2 \rho_0 c \, U_0 \left| \sin\left( \tfrac{k}{2} \left( \sqrt{z^2 + a^2} - z \right) \right) \right| $$

where $a$ is the piston radius and $z$ the distance from the face. The largest possible on-axis
pressure is therefore $2 \rho_0 c U_0$, about 2.95 MPa for water at 1 m/s.
The number of point sources per wavelength controls how well the sum approximates the integral.

> **To expand (Yash):** derive the on-axis formula from the Rayleigh integral; explain the near field / far field boundary (Rayleigh distance $a^2/\lambda$) and why there is only one on-axis maximum here ($ka \approx 4.2$).

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "02_piston_source"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. Parameters

In [ ]:
frequency = 1000.0                 # Hz
piston_radius = 1.0                # m
piston_location = (-4.0, 0.0, 0.0) # centre of the piston face
piston_axis = (1, 0, 0)
velocity = 1.0                     # m/s, normal velocity of the face
element_size = 0.4                 # m, mesh of the bone sphere
grid_resolution = (100, 75) if CI_MODE else (240, 180)
bounding_box = (-3, 5, -3, 3)

## 2. Source, geometry and materials

In [ ]:
source = optimus.source.create_piston(frequency, radius=piston_radius, location=piston_location,
                                      source_axis=piston_axis, velocity=velocity)
geometry = optimus.geometry.shapes.Sphere(element_size=element_size)
material_ext = optimus.material.load_material("water")
material_int = optimus.material.load_material("bone (cortical)")

## 3. Incident field on the axis against the analytical formula

Before any BEM solve, compare the source on its own with the closed-form result.
The comparison is repeated with 24 point sources per wavelength to show the source discretisation converging.

In [ ]:
rho, c = material_ext.density, material_ext.speed_of_sound
k = 2 * np.pi * frequency / c
z = np.linspace(0.05, 6.0, 120)        # distance from the piston face, m
axis_points = np.vstack((piston_location[0] + z, 0 * z, 0 * z))
p_analytical = 2 * rho * c * velocity * np.abs(np.sin(k / 2 * (np.sqrt(z**2 + piston_radius**2) - z)))

def on_axis_error(points_per_wavelength):
    s = optimus.source.create_piston(frequency, radius=piston_radius, location=piston_location,
                                     source_axis=piston_axis, velocity=velocity,
                                     number_of_point_sources_per_wavelength=points_per_wavelength)
    p = np.abs(s.pressure_field(material_ext, axis_points))
    return p, np.max(np.abs(p - p_analytical)) / np.max(p_analytical)

p_default, error_default = on_axis_error(6)
p_fine, error_fine = on_axis_error(24)
RESULTS["on_axis_error_6ppw"] = round(float(error_default), 4)
RESULTS["on_axis_error_24ppw"] = round(float(error_fine), 4)
print("Max on-axis error: %.3f (6 per wavelength), %.3f (24 per wavelength)" % (error_default, error_fine))

fig = plt.figure(figsize=(7, 4))
plt.plot(z, p_analytical / 1e6, "k-", label="analytical (Kinsler et al. 7.4)")
plt.plot(z, p_default / 1e6, "o", ms=3, label="OptimUS, 6 per wavelength")
plt.plot(z, p_fine / 1e6, "x", ms=3, label="OptimUS, 24 per wavelength")
plt.xlabel("distance from piston face z [m]")
plt.ylabel("|p| [MPa]")
plt.legend()
plt.grid(alpha=0.3)
save_figures(fig, NOTEBOOK_NAME + "_axis")
plt.show()

## 4. Model and solve

In [ ]:
model = optimus.model.create_default_model(source, geometry, material_ext, material_int)
t0 = time.time()
model.solve()
RESULTS["solve_time_s"] = round(time.time() - t0, 2)
iterations, converged = gmres_converged(model)
RESULTS["gmres_iterations"] = int(iterations)
print("GMRES iterations:", iterations)

## 5. Field on the plane z = 0

In [ ]:
plane = optimus.postprocess.VisualisePlane(model)
plane.create_computational_grid(resolution=grid_resolution, bounding_box=bounding_box)
plane.compute_fields()
figs = optimus.postprocess.plot_pressure_field(plane, field="incident")
save_figures(figs, NOTEBOOK_NAME + "_incident")
figs = optimus.postprocess.plot_pressure_field(plane, field="total")
save_figures(figs, NOTEBOOK_NAME + "_total")

## 6. Checks

Limits are measured values plus a margin (measured 9 Nov 2026 in the Docker image):
6.6% on-axis error at 6 point sources per wavelength and 1.2% at 24. The default source is
accepted up to 10%, the 24 per wavelength source up to 3%, and the finer source must be more accurate.
The largest error is right at the piston face, where the point sources are closest.

In [ ]:
check("source_default_accuracy", error_default < 0.10,
      "on-axis error %.4f at 6 per wavelength, limit 0.10" % error_default)
check("source_converges", error_fine < 0.03 and error_fine < error_default,
      "on-axis error %.4f at 24 per wavelength, limit 0.03" % error_fine)
check("gmres_converged", converged, "%d iterations" % iterations)
total = plane.total_field_imshow
check("field_is_finite", np.all(np.isfinite(total)),
      "%d NaN or inf values on the plane" % np.sum(~np.isfinite(total)))

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))